# 00 — Common Utilities (shared library)

Reusable building blocks for the MINDX-Mart pipeline. The Bronze / Silver / Gold notebooks pull
these in with `%run 00_common_utils` so the logic lives in **one** place.

Provides:
- **Config loader** — reads `Files/config/pipeline_config.json` (single source of truth).
- **Structured logger** + **audit run-log** (`audit_pipeline_run_log`).
- **Data-quality engine** — applies the config rules, splits clean/quarantine, logs `audit_dq_results`.
- **Delta helpers** — idempotent `merge_upsert`, `table_exists`.
- **Dimension helpers** — surrogate keys, unknown member, `scd1_upsert`, `scd2_upsert`.
- **Calendar generator** and **maintenance** (OPTIMIZE / ZORDER / VACUUM).

In [ ]:
import json, uuid, datetime as _dt
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- Config loader (single source of truth) ---
_CONFIG_FILE = "Files/config/pipeline_config.json"
try:
    with open(f"/lakehouse/default/{_CONFIG_FILE}", "r", encoding="utf-8") as _fh:
        CONFIG = json.load(_fh)
except FileNotFoundError:
    import notebookutils
    CONFIG = json.loads(notebookutils.fs.head(_CONFIG_FILE, 5 * 1024 * 1024))

AUDIT_RUN_LOG = CONFIG["audit"]["run_log_table"]
AUDIT_DQ = CONFIG["audit"]["dq_results_table"]
print(f"Config v{CONFIG['version']} loaded for project '{CONFIG['project']}'.")

In [ ]:
def log(msg, level="INFO"):
    ts = _dt.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] [{level}] {msg}")

def table_exists(name: str) -> bool:
    return spark.catalog.tableExists(name)

def new_batch_id() -> str:
    """A batch id ties Bronze/Silver/Gold rows from the same orchestration run together."""
    return _dt.datetime.utcnow().strftime("%Y%m%d%H%M%S")

## Audit run-log

In [ ]:
_RUN_LOG_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("batch_id", T.StringType()),
    T.StructField("layer", T.StringType()),
    T.StructField("entity", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("rows_read", T.LongType()),
    T.StructField("rows_written", T.LongType()),
    T.StructField("rows_quarantined", T.LongType()),
    T.StructField("start_ts", T.TimestampType()),
    T.StructField("end_ts", T.TimestampType()),
    T.StructField("duration_sec", T.DoubleType()),
    T.StructField("message", T.StringType()),
])

def start_run(layer: str, entity: str, batch_id: str) -> dict:
    run = {"run_id": str(uuid.uuid4()), "batch_id": batch_id, "layer": layer,
           "entity": entity, "start": _dt.datetime.utcnow()}
    log(f"START {layer}/{entity} run_id={run['run_id']} batch={batch_id}")
    return run

def end_run(run: dict, status: str, rows_read=0, rows_written=0, rows_quarantined=0, message=""):
    end = _dt.datetime.utcnow()
    row = (run["run_id"], run["batch_id"], run["layer"], run["entity"], status,
           int(rows_read), int(rows_written), int(rows_quarantined),
           run["start"], end, (end - run["start"]).total_seconds(), message)
    (spark.createDataFrame([row], _RUN_LOG_SCHEMA)
        .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(AUDIT_RUN_LOG))
    log(f"END   {run['layer']}/{run['entity']} status={status} "
        f"read={rows_read} written={rows_written} quarantined={rows_quarantined}")

## Data-quality engine (config-driven)
Each rule's `expression` is evaluated with `F.expr`. Rows failing any **REJECT** rule are quarantined;
**WARN** rules are recorded but the row is kept. Per-rule failure counts are logged to `audit_dq_results`.

In [ ]:
def apply_dq(df, rules):
    """Annotate df with _dq_failed_reject / _dq_failed_warn / _is_clean / quarantine_reason."""
    reject = [r for r in rules if r["severity"] == "REJECT"]
    warn = [r for r in rules if r["severity"] == "WARN"]

    def reasons(rule_set):
        arr = F.array(*[F.when(~F.expr(r["expression"]), F.lit(r["reason"])) for r in rule_set]) \
              if rule_set else F.array().cast("array<string>")
        return F.filter(arr, lambda x: x.isNotNull())

    out = (df
        .withColumn("_dq_failed_reject", reasons(reject))
        .withColumn("_dq_failed_warn", reasons(warn)))
    out = (out
        .withColumn("_is_clean", F.size("_dq_failed_reject") == 0)
        .withColumn("quarantine_reason",
                    F.when(F.size("_dq_failed_reject") > 0, F.col("_dq_failed_reject")[0])))
    return out

def log_dq_results(df_eval, rules, entity, run, total=None):
    """Write per-rule failure counts to the DQ results table."""
    total = total if total is not None else df_eval.count()
    rows = []
    for r in rules:
        failed = df_eval.where(~F.expr(r["expression"])).count()
        rows.append((run["run_id"], run["batch_id"], entity, r["rule_id"], r["column"],
                     r["severity"], r["reason"], int(failed), int(total),
                     _dt.datetime.utcnow()))
    schema = T.StructType([
        T.StructField("run_id", T.StringType()), T.StructField("batch_id", T.StringType()),
        T.StructField("entity", T.StringType()), T.StructField("rule_id", T.StringType()),
        T.StructField("column", T.StringType()), T.StructField("severity", T.StringType()),
        T.StructField("reason", T.StringType()), T.StructField("failed_count", T.LongType()),
        T.StructField("total_count", T.LongType()), T.StructField("logged_at", T.TimestampType()),
    ])
    (spark.createDataFrame(rows, schema)
        .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(AUDIT_DQ))

## Delta helpers — idempotent upsert

In [ ]:
def merge_upsert(df, table: str, keys: list, partition_by=None):
    """Idempotent MERGE: update matched rows, insert new ones (keyed on `keys`).
    Re-running the same batch is a no-op."""
    if not table_exists(table):
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if partition_by:
            writer = writer.partitionBy(*partition_by)
        writer.saveAsTable(table)
        return
    cond = " AND ".join([f"t.{k} = s.{k}" for k in keys])
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

## Dimension helpers — surrogate keys, unknown member, SCD1 / SCD2

In [ ]:
_NULL_TOKEN = "__NULL__"

def _row_hash(cols):
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit(_NULL_TOKEN)) for c in cols]), 256)

def add_surrogate_key(df, sk_col, start=1):
    w = Window.orderBy(F.monotonically_increasing_id())
    return df.withColumn(sk_col, (F.row_number().over(w) + F.lit(int(start) - 1)).cast("long"))

def _current_max_sk(table, sk_col):
    m = spark.table(table).agg(F.max(sk_col)).first()[0]
    return int(m) if m is not None else 0

def add_unknown_member(table, sk_col, biz_keys):
    """Insert the -1 'Unknown' member so fact->dim joins never drop rows."""
    df = spark.table(table)
    if df.where(F.col(sk_col) == -1).limit(1).count() > 0:
        return
    vals = {c: None for c in df.columns}
    vals[sk_col] = -1
    for k in biz_keys:
        vals[k] = "UNKNOWN"
    if "is_current" in vals:
        vals["is_current"] = True
    from pyspark.sql import Row
    spark.createDataFrame([Row(**{c: vals[c] for c in df.columns})], df.schema) \
        .write.format("delta").mode("append").saveAsTable(table)

In [ ]:
def scd1_upsert(incoming, table, sk_col, biz_keys, tracked_cols, batch_id):
    """Type-1 dimension: overwrite changed attributes, keep one row per business key."""
    incoming = incoming.dropDuplicates(biz_keys).withColumn("_batch_id", F.lit(batch_id))
    if not table_exists(table):
        init = add_surrogate_key(incoming, sk_col, 1)
        init.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
        add_unknown_member(table, sk_col, biz_keys)
        return
    existing = spark.table(table)
    new_keys = incoming.join(existing.select(*biz_keys), biz_keys, "left_anti")
    if new_keys.limit(1).count() > 0:
        to_insert = add_surrogate_key(new_keys, sk_col, _current_max_sk(table, sk_col) + 1)
        to_insert.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)
    if tracked_cols and set(tracked_cols) - set(biz_keys):
        cond = " AND ".join([f"t.{k} = s.{k}" for k in biz_keys])
        (DeltaTable.forName(spark, table).alias("t").merge(incoming.alias("s"), cond)
            .whenMatchedUpdate(set={c: f"s.{c}" for c in tracked_cols}).execute())

def scd2_upsert(incoming, table, sk_col, biz_keys, tracked_cols, batch_id):
    """Type-2 dimension: keep full history. Changed attributes close the current
    row (is_current=false, valid_to=now) and open a new versioned row."""
    incoming = incoming.dropDuplicates(biz_keys).withColumn("_row_hash", _row_hash(tracked_cols))
    now = F.current_timestamp()
    open_cols = (lambda d: d.withColumn("valid_from", now)
                            .withColumn("valid_to", F.lit(None).cast("timestamp"))
                            .withColumn("is_current", F.lit(True))
                            .withColumn("_batch_id", F.lit(batch_id)))
    if not table_exists(table):
        init = add_surrogate_key(open_cols(incoming), sk_col, 1)
        init.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
        add_unknown_member(table, sk_col, biz_keys)
        return
    dt = DeltaTable.forName(spark, table)
    current = dt.toDF().where("is_current = true").select(*biz_keys, F.col("_row_hash").alias("_cur_hash"))
    joined = incoming.alias("s").join(current.alias("c"), biz_keys, "left")
    changed = joined.where(F.col("_cur_hash").isNull() | (F.col("_cur_hash") != F.col("s._row_hash")))
    if changed.limit(1).count() == 0:
        return  # nothing changed -> idempotent no-op
    new_versions = add_surrogate_key(open_cols(changed.select("s.*")), sk_col,
                                     _current_max_sk(table, sk_col) + 1)
    # close the existing current rows whose attributes changed
    changed_existing = changed.where(F.col("_cur_hash").isNotNull()).select(*[F.col(f"s.{k}").alias(k) for k in biz_keys]).distinct()
    if changed_existing.limit(1).count() > 0:
        cond = " AND ".join([f"t.{k} = k.{k}" for k in biz_keys])
        (dt.alias("t").merge(changed_existing.alias("k"), cond)
            .whenMatchedUpdate(condition="t.is_current = true",
                               set={"is_current": "false", "valid_to": "current_timestamp()"}).execute())
    new_versions.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)

## Calendar generator & maintenance

In [ ]:
def generate_calendar(start_date: str, end_date: str):
    base = spark.sql(
        f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS full_date")
    return (base
        .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
        .withColumn("year", F.year("full_date"))
        .withColumn("quarter", F.quarter("full_date"))
        .withColumn("month", F.month("full_date"))
        .withColumn("month_name", F.date_format("full_date", "MMMM"))
        .withColumn("day", F.dayofmonth("full_date"))
        .withColumn("day_of_week", F.dayofweek("full_date"))
        .withColumn("day_name", F.date_format("full_date", "EEEE"))
        .withColumn("week_of_year", F.weekofyear("full_date"))
        .withColumn("year_month", F.date_format("full_date", "yyyy-MM"))
        .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7)))

def optimize_table(table, zorder_cols=None):
    try:
        if zorder_cols:
            spark.sql(f"OPTIMIZE {table} ZORDER BY ({', '.join(zorder_cols)})")
        else:
            spark.sql(f"OPTIMIZE {table}")
        log(f"OPTIMIZE {table} done")
    except Exception as e:
        log(f"OPTIMIZE {table} skipped: {e}", "WARN")

def vacuum_table(table, retain_hours=168):
    try:
        spark.sql(f"VACUUM {table} RETAIN {retain_hours} HOURS")
        log(f"VACUUM {table} done")
    except Exception as e:
        log(f"VACUUM {table} skipped: {e}", "WARN")

log("Common utilities loaded.")